###Ingestion del archivo "genre.csv"

####Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
genres_schema = StructType(fields=[
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])

In [0]:
genres_df = spark.read \
    .option("header", True) \
    .schema(genres_schema) \
    .csv(f"abfss://bronze@moviehistory8.dfs.core.windows.net/{v_file_date}/genre.csv")

####Paso 2 - Cambiar el nombre de las columnas segun lo requerido

In [0]:
genres_renamed_df = genres_df.withColumnRenamed("genreId", "genre_id") \
    .withColumnRenamed("genreName", "genre_name")

####Paso 3 - Agregar las columnas "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
genres_final_df = genres_renamed_df.withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Production")) \
    .withColumn("file_date", lit(v_file_date))

###Paso 4 - Escribir datos en el DataLake en formato "Parquet"

In [0]:
#genres_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistory8.dfs.core.windows.net/genres")

In [0]:
genres_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
%sql
SELECT * FROM movie_silver.genres;

genre_id,genre_name,ingestion_date,enviroment,file_date
12,Adventure,2026-09-26T16:17:49.556149Z,Production,2024-12-16
14,Fantasy,2026-09-26T16:17:49.556149Z,Production,2024-12-16
16,Animation,2026-09-26T16:17:49.556149Z,Production,2024-12-16
18,Drama,2026-09-26T16:17:49.556149Z,Production,2024-12-16
27,Horror,2026-09-26T16:17:49.556149Z,Production,2024-12-16
28,Action,2026-09-26T16:17:49.556149Z,Production,2024-12-16
35,Comedy,2026-09-26T16:17:49.556149Z,Production,2024-12-16
36,History,2026-09-26T16:17:49.556149Z,Production,2024-12-16
37,Western,2026-09-26T16:17:49.556149Z,Production,2024-12-16
53,Thriller,2026-09-26T16:17:49.556149Z,Production,2024-12-16


In [0]:
#%fs
#ls abfss://silver@moviehistory8.dfs.core.windows.net/genres

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistory8.dfs.core.windows.net/genres"))

In [0]:
dbutils.notebook.exit("Success")